In [84]:
#imports
import os
import json
from dotenv import load_dotenv
import gradio as gr
from openai import OpenAI
import sqlite3

In [94]:
load_dotenv(override=True)
gemini_api_key = os.getenv("GOOGLE_API_KEY")
Model = "gemini-3.1-flash"
gemini = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta/openai/",api_key=gemini_api_key)
ollama = OpenAI(base_url="http://localhost:11434/v1",api_key="ollama")

In [95]:
DB = "carrental.db"
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
    CREATE TABLE IF NOT EXISTS cars(
    car_type TEXT PRIMARY KEY,
    price_per_day REAL
    )
    ''')
    conn.commit()

In [96]:
car_data = [('sedan',40),('suv',70),('truck',90),('convertible',120)]
with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.executemany(
        "INSERT OR IGNORE INTO cars (car_type,price_per_day) VALUES (?,?)",car_data
    )
    conn.commit()

In [97]:
def get_car_price(car_type,days):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('Select price_per_day FROM cars where car_type = ?',(car_type.lower(),))
        result = cursor.fetchone()
    
    if result :
        total = result[0]*days
        return f"A {car_type} costs ${result[0]}/day. for {days} days that's {total} total."
    else:
        return f"Sorry we dont have the {car_type} available."

In [98]:
get_car_price_function = {
    "name": "get_car_price",
    "description":"Get the rental price for a specific car type for a given number",
    "parameters":{
        "type":"object",
        "properties":{
            "car_type":{
                "type":"string",
                "description":"The type of car e.g. sedan,suv,truck,convertible"
            },
            "days":{
                "type":"integer",
                "description":"The number of days customer wants to rent the car"
            }
        },
        "required":["car_type","days"],
        "additionalProperties":False
    }
}

tools = [{"type":"function","function":get_car_price_function}]

In [99]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_car_price":
            arguments = json.loads(tool_call.function.arguments)
            car_type = arguments.get("car_type")
            days = arguments.get("days")
            result = get_car_price(car_type,days)
            responses.append(
                {
                    "role":"tool",
                    "content":result,
                    "tool_call_id":tool_call.id
                }
            )
    return responses

In [100]:
system_prompt = """
You are a AI Assistant at a Car Rental called DriveEasy.
Give short, courteous answers, no more than 1 sentance.
Always be Accuraate. If you dont know the answer, say so.
"""

In [106]:
def chat(message,history):
    history = [{"role":h["role"],"content":h["content"]} for h in history]
    messages = [{"role":"system","content":system_prompt}]+history+[{"role":"user","content":message}]
    response = ollama.chat.completions.create(
        model="gemma3",
        messages=messages,
        tools = tools
    )
    while response.choices[0].finish_reason == "tool_calls" :
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model="gemma3",messages=messages,tools=tools)

    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7870
* To create a public link, set `share=True` in `launch()`.


c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
Traceback (most recent call last):
  File "c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\queueing.py", line 867, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<5 lines>...
    )
    ^
  File "c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\route_utils.py", line 386, in call_process_api
    output = await app.get_blocks().process_api(
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
    ...<11 lines>...
    )
    ^
  File "c:\Users\AKPGX7\AppData\Local\Programs\Python\Python313\Lib\site-packages\gradio\blocks.py", line 2280, in process_api
    result = await self.call_